# Task 2.1 — Feature audit

This notebook audits the HDF5 data product produced by the final C++ extractor. It works on the 7k-row test file or the full ~1.4M-row file.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/python')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.feature_audit import *

In [2]:
# Set these paths before running.
H5_PATHS = [
    '/work/clas12b/users/skuditha/ALERT/alert_pid/data/small.h5',
    #'PROJECT_ROOT/data/training_sample.h5',
]
LABEL_MAP_PATH = '/work/clas12b/users/skuditha/ALERT/alert_pid/config/label_map.json'
OUTDIR = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/reports/feature_audit')

print('Edit H5_PATHS first.')

Edit H5_PATHS first.


In [3]:
# Load dataset
ds = load_audit_dataset(H5_PATHS, LABEL_MAP_PATH)
print({'n_rows': ds.n_rows, 'n_features': ds.n_features, 'files': [str(p) for p in ds.paths]})
print(ds.feature_names)

{'n_rows': 4987, 'n_features': 38, 'files': ['/work/clas12b/users/skuditha/ALERT/alert_pid/data/small.h5']}
['px', 'py', 'pz', 'p', 'pt', 'theta', 'phi', 'vx', 'vy', 'vz', 'vr', 'v3', 'n_hits', 'sum_adc', 'path', 'dEdx', 'dedx_recomputed', 'p_drift', 'sum_residuals', 'residual_per_hit', 'adc_per_hit', 'tof_time', 'pathlength', 'cluster_x', 'cluster_y', 'cluster_z', 'cluster_energy', 'n_bar', 'n_wedge', 'beta', 'm2', 'log_p', 'log_pt', 'log_sum_adc', 'log_path', 'log_dEdx', 'log_dedx_recomputed', 'log_cluster_energy']


In [4]:
class_balance = compute_class_balance(ds)
feature_summary = compute_feature_summary(ds)
mask_summary = compute_mask_summary(ds)
unit_sanity = infer_unit_sanity(ds)
pathologies = detect_pathologies(ds)
separation = compute_separation_table(ds)
pair_focus = pair_focus_summary(ds)

class_balance

,class_index,class_name,count,fraction
0,0,proton,739,0.148185
1,1,deuteron,903,0.181071
2,2,triton,931,0.186685
3,3,helium3,1222,0.245037
4,4,helium4,1192,0.239021


In [5]:
feature_summary.head(20)

,feature,valid_count,invalid_count,valid_fraction,raw_min,raw_max,valid_min,valid_max,valid_mean,valid_std,zeros_in_stored_values
0,m2,4871,116,0.97674,0.000000,3.265852e+08,445.906525,3.265852e+08,9.888985e+06,2.047569e+07,116
1,adc_per_hit,4987,0,1.00000,33.500000,3.789778e+03,33.500000,3.789778e+03,8.653899e+02,6.601383e+02,0
2,beta,4987,0,1.00000,0.048402,1.196511e+00,0.048402,1.196511e+00,3.636530e-01,2.277309e-01,0
3,cluster_energy,4987,0,1.00000,0.393653,3.009937e+01,0.393653,3.009937e+01,8.096868e+00,6.350446e+00,0
4,cluster_x,4987,0,1.00000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,-1.637770e+00,6.277684e+01,0
5,cluster_y,4987,0,1.00000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,1.147974e+00,6.224570e+01,0
6,cluster_z,4987,0,1.00000,-297.909119,3.136903e+02,-297.909119,3.136903e+02,-3.231126e+00,8.293153e+01,0
7,dEdx,4987,0,1.00000,1.194194,5.454643e+02,1.194194,5.454643e+02,8.078635e+01,6.944698e+01,0
8,dedx_recomputed,4987,0,1.00000,1.194194,5.454643e+02,1.194194,5.454643e+02,8.078635e+01,6.944698e+01,0
9,log_cluster_energy,4987,0,1.00000,-0.932286,3.404504e+00,-0.932286,3.404504e+00,1.662548e+00,1.053731e+00,0


In [6]:
mask_summary

,metric,value
0,rows_with_any_masked_feature,116.00000
1,rows_with_no_masked_feature,4871.00000
2,mean_invalid_features_per_row,0.02326
3,max_invalid_features_in_row,1.00000


In [7]:
unit_sanity

,check,value,comment
0,p_median,595.028809,"Large O(1) suggests GeV/c, O(100-1000) suggest..."
1,tof_time_median_ns,1.323250,Expected ns-scale positive cluster timing.
2,pathlength_median,114.017540,Check whether pathlength looks mm-scale rather...
3,beta_median_stored,0.292162,Should be comfortably below 1 for most rows.
4,beta_median_recomputed_mm_ns,0.292162,Recomputed with c = 299.792458 mm/ns.
5,frac_beta_gt_1p0_stored,0.023260,Diagnostic only; no row cuts in audit.
6,frac_beta_gt_1p0_recomputed,0.023260,High value flags a unit mismatch or timing pat...
7,frac_m2_negative,0.000000,"Negative m2 can occur, but large fractions des..."


In [8]:
pathologies

,pathology,count
0,nonfinite_stored_values,0
1,rows_with_any_nonfinite_stored_value,0
2,valid_time_le_zero,0
3,valid_pathlength_le_zero,0
4,valid_p_le_zero,0
5,valid_dEdx_le_zero,0
6,valid_cluster_energy_le_zero,0
7,valid_beta_le_zero,0
8,valid_beta_gt_1p2,0


In [9]:
separation.head(15)

,feature,fisher_score
0,log_sum_adc,1.795703
1,log_dedx_recomputed,1.584114
2,log_dEdx,1.584114
3,adc_per_hit,1.427591
4,cluster_energy,1.353749
5,sum_adc,1.206503
6,log_cluster_energy,1.136702
7,dEdx,0.912297
8,dedx_recomputed,0.912297
9,residual_per_hit,0.080849


In [10]:
pair_focus

,feature,class_name,count,median,p16,p84
0,p,deuteron,903,6.505302e+02,423.190427,1.146489e+03
1,p,helium4,1192,5.705489e+02,375.378325,9.207345e+02
2,tof_time,deuteron,903,1.304626e+00,0.677642,2.137255e+00
3,tof_time,helium4,1192,1.463123e+00,0.783752,2.251768e+00
4,pathlength,deuteron,903,1.140175e+02,91.082382,1.548419e+02
5,pathlength,helium4,1192,1.140175e+02,91.082382,1.548419e+02
6,dEdx,deuteron,903,2.637278e+01,11.134026,6.053500e+01
7,dEdx,helium4,1192,1.283544e+02,78.705803,2.018648e+02
8,cluster_energy,deuteron,903,3.406907e+00,1.044147,5.810862e+00
9,cluster_energy,helium4,1192,1.438648e+01,7.479401,1.952906e+01


In [11]:
corr = compute_correlation_matrix(ds)
high_corr = high_correlation_pairs(corr, threshold=0.95)
high_corr.head(30)

,feature_a,feature_b,corr
0,dEdx,dedx_recomputed,1.000000
1,log_dEdx,log_dedx_recomputed,1.000000
2,p,p_drift,0.999998
3,sum_residuals,residual_per_hit,0.984828
4,sum_adc,adc_per_hit,0.980943
5,log_sum_adc,log_dedx_recomputed,0.951396
6,log_sum_adc,log_dEdx,0.951396


In [12]:
OUTDIR.mkdir(parents=True, exist_ok=True)
plot_feature_histograms(ds, KEY_PHYSICS_FEATURES, OUTDIR / 'key_histograms')
plot_feature_histograms(ds, DEFAULT_FEATURE_NAMES, OUTDIR / 'histograms')
plot_scatter_by_class(ds, 'p', 'beta', OUTDIR / 'beta_vs_p.png')
plot_scatter_by_class(ds, 'p', 'm2', OUTDIR / 'm2_vs_p.png')
plot_scatter_by_class(ds, 'p', 'dEdx', OUTDIR / 'dEdx_vs_p.png')
plot_scatter_by_class(ds, 'pathlength', 'cluster_energy', OUTDIR / 'cluster_energy_vs_pathlength.png')
plot_correlation_heatmap(corr, OUTDIR / 'correlation_heatmap.png')
print(f'Plots written under {OUTDIR.resolve()}')

Plots written under /ceph24/hallb/clas12/users/skuditha/ALERT/alert_pid/reports/feature_audit


In [20]:
# One-shot batch run
# tables = run_full_feature_audit(H5_PATHS, LABEL_MAP_PATH, OUTDIR)